In [2]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait # Importação Nova
from selenium.webdriver.support import expected_conditions as EC # Importação Nova
import time
import random
import os
import re

In [ ]:
# ==============================================================================
# BLOCO 1: Configuração do Robô
# ==============================================================================

# O 'Service' baixa automaticamente a versão correta driver do Chrome
servico = Service(ChromeDriverManager().install())

# As 'Options' são as configurações do navegador
options = webdriver.ChromeOptions()

# Começar tela cheia ajuda a evitar erros de layout
options.add_argument("--start-maximized")

# --- O DISFARCE (Essencial para sites com Cloudflare/Anti-Robô) ---
options.add_argument("--disable-blink-features=AutomationControlled") 
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)

# Inicia o navegador
navegador = webdriver.Chrome(service=servico, options=options)

# ==============================================================================
# BLOCO 2: SITE RECLAME AQUI E SEU OBJETIVO
# ==============================================================================
lista_total = []
base_url = "https://www.reclameaqui.com.br/empresa/banco-master/lista-reclamacoes/"

try:
    print("--- INICIANDO RASPAGEM DE DADOS ---")
    
    # 1. Navegar para o site
    navegador.get(base_url)

    # PAUSA PARA RESOLVER O CAPTCHA
    print("\n" + "="*60)
    print("🚨 AÇÃO NECESSÁRIA! 🚨")
    print("1. Vá no Chrome e resolva o 'Sou Humano' se aparecer.")
    print("2. Espere a lista carregar.")
    print("3. VOLTE AQUI E APERTE [ENTER].")
    print("="*60 + "\n")
    input("Estou aguardando... Aperte Enter quando estiver pronto: ")
    
    print("Robô assumindo o controle...")

    corpo_pagina = navegador.find_element(By.TAG_NAME, 'body').text
    
    # Regex para achar "X reclamações" ou números soltos grandes
    match = re.search(r'(\d{1,3}(?:\.\d{3})*) reclamações', corpo_pagina)
    
    if match:
        total_str = match.group(1).replace('.', '')
        total_reclamacoes = int(total_str)
        total_paginas = (total_reclamacoes // 10) + 1
        
        print(f"📊 TOTAL ENCONTRADO: {total_reclamacoes} reclamações.")
        print(f"📚 Total de Páginas: {total_paginas}")
        
        # Define um limite seguro (Ex: Pegar no máximo 200 páginas ou o total real, o que for menor)
        limite_paginas = min(200, total_paginas)
        print(f"🎯 Meta do Robô: Ler {limite_paginas} páginas (Aprox. {limite_paginas*10} itens).")
        
    else:
        print("⚠️ Não achei o número total. Vou usar o padrão de 20 páginas.")
        limite_paginas = 20

    # Atualiza o loop com o novo limite
    for pagina in range(1, limite_paginas + 1):

    # Só navega se não for a página 1 (que já estamos nela pós-captcha)
        if pagina > 1:
            print(f'Indo para a página {pagina}...')
            navegador.get(f"{base_url}?pagina={pagina}")
            # Pausa humana
            time.sleep(random.uniform(3, 5))
        
        # ==========================================================================
        # BLOCO 3: A INTERAÇÃO
        # ==========================================================================
        
        # SCROLL: Rolar a página é gatilho para carregar dados em sites modernos
        navegador.execute_script("window.scrollTo(0, document.body.scrollHeight/2);")
        
        try:
            # ESPERA INTELIGENTE (WebDriverWait)
            # Diferente do time.sleep, este comando vigia a tela.
            # "Espere até 15s para aparecerem links que tenham '/banco-master/' no endereço"
            wait = WebDriverWait(navegador, 15)
            elementos = wait.until(
                EC.presence_of_all_elements_located((By.CSS_SELECTOR, "a[href*='/banco-master/']"))
            )
            
            # EXTRAÇÃO (Loop pelos itens da tela)
            # Recaptura os elementos para garantir que o site não atualizou e quebrou o link
            elementos_atualizados = navegador.find_elements(By.CSS_SELECTOR, "a[href*='/banco-master/']")
            
            contador = 0
            for item in elementos_atualizados:
                texto = item.text
                link = item.get_attribute('href')
                
                # Filtro de Qualidade: Ignora botões de menu e duplicatas
                if len(texto) > 20 and link not in [x['link'] for x in lista_total]:
                    lista_total.append({
                        'banco': 'Banco Master',
                        'titulo': texto,
                        'link': link,
                        'pagina': pagina
                    })
                    contador += 1
            
            print(f"✅ Página {pagina}: {contador} itens.")
            
        except Exception as e:
            print(f"⚠️ Falha na página {pagina}. O site demorou ou bloqueou.")
            # Não para o código, tenta a próxima página
            continue 

    # ==========================================================================
    # BLOCO 4: Salvar para o Dataframe
    # ==========================================================================
    if lista_total:
            df_ra = pd.DataFrame(lista_total)
            
            # Alerta para reclamações
            palavra_chave = ['saque', 'dinheiro', 'travado', 'bloqueio', 'sumiu', 'golpe', 'indevido']
            df_ra['alerta'] = df_ra['titulo'].str.lower().apply(lambda x: any(k in x for k in palavra_chave))
            
            # Salva o arquivo completo na pasta RAW
            df_ra.to_parquet("../dados/raw/reclamacoes_master.parquet", index=False)
            
            print(f"\n🎯 SUCESSO! {len(df_ra)} reclamações salvas com a coluna 'alerta'.")
            
            # Amostra crítica
            if df_ra['alerta'].sum() > 0:
                print(f"🔥 Sinais de Risco: {df_ra['alerta'].sum()}")
                display(df_ra[df_ra['alerta']==True].head())

except Exception as e:
    print(f"Erro Crítico: {e}")

finally:
    # Fecha o navegador
    if 'navegador' in locals():
        navegador.quit()
    print("Fim do processo.")

--- INICIANDO RASPAGEM DE DADOS ---


In [3]:
caminho_raw = "../dados/raw"

df_reclamacao = pd.read_parquet(os.path.join(caminho_raw, "reclamacoes_master.parquet"))

print(f'O dataframe possui {df_reclamacao.shape[0]} linhas e {df_reclamacao.shape[1]} colunas')

df_reclamacao

O dataframe possui 465 linhas e 5 colunas


,banco,titulo,link,pagina,alerta
0,Banco Master,Banco Master não libera saldo devedor para por...,https://www.reclameaqui.com.br/banco-master/ba...,1,False
1,Banco Master,"R$ 119,65 R$2 .363,81. R$ NEGOCIE PELO WHATSAP...",https://www.reclameaqui.com.br/banco-master/r-...,1,False
2,Banco Master,Solicitação de boleto para amortização de cons...,https://www.reclameaqui.com.br/banco-master/so...,1,False
3,Banco Master,Dificuldade em realizar saque de cartão de cré...,https://www.reclameaqui.com.br/banco-master/di...,1,True
4,Banco Master,Cartão do empréstimo não funciona e valor não ...,https://www.reclameaqui.com.br/banco-master/ca...,1,False
...,...,...,...,...,...
460,Banco Master,Não envio do boleto de quitação e cobrança ind...,https://www.reclameaqui.com.br/banco-master/na...,49,False
461,Banco Master,Banco não libera margem para cliente: Denúncia...,https://www.reclameaqui.com.br/banco-master/ba...,49,False
462,Banco Master,Empresa com práticas abusivas e dificuldades p...,https://www.reclameaqui.com.br/banco-master/em...,49,False
463,Banco Master,Cobrança indevida na fatura do cartão de crédi...,https://www.reclameaqui.com.br/banco-master/co...,49,False
